# Download entirety of the Datasets

In [8]:
import os
import numpy as np
import pandas as pd
import requests
import yfinance as yf

In [9]:
# Remove start_date / end_date (we will pull max history)
output_dir = "../data/sp500_individual_gbm/"
os.makedirs(output_dir, exist_ok=True)

MIN_YEARS = 40
MIN_TRADING_DAYS = MIN_YEARS * 252  # simple rule-of-thumb

In [10]:
import re

def get_sp500_tickers(exclude_problematic=True):
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers)
    table = pd.read_html(response.text)
    df = table[0]

    tickers = df["Symbol"].astype(str).tolist()

    if exclude_problematic:
        # Keep only tickers like AAPL, BRK, GOOG, etc. (no '.', '-', '/', etc.)
        tickers = [t for t in tickers if re.fullmatch(r"[A-Z0-9]+", t)]

    return tickers

In [11]:
def transformation(data):
        # YFinance indexes by Date automatically; ensure it is sorted
        data = data.sort_index()
        
        # Calculate log-returns: r_t = ln(P_t / P_{t-1}) [cite: 5353]
        ohlc_cols = ['Open', 'High', 'Low', 'Close']
        # Divide OHLC by previous day's close to get relative returns
        prev_close = data['Close'].shift(1)
        ohlc_log_rets = np.log(data[ohlc_cols].div(prev_close, axis=0))
        
        # Volume log-returns (adding 1 to avoid log(0))
        volume_log_rets = np.log(data['Volume'] + 1) - np.log(data['Volume'].shift(1) + 1)

        processed_data = pd.concat([ohlc_log_rets, volume_log_rets], axis=1).dropna()
        data_arr = processed_data.values.astype(np.float32)

        # Global Standardization: r_std = (r - mu) / sigma [cite: 5370]
        mean = data_arr.mean(axis=0)
        std = data_arr.std(axis=0)
        data_standardized = (data_arr - mean) / (std + 1e-8)
        
        return data_standardized, processed_data.index

In [12]:
tickers = get_sp500_tickers()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4936\631958006.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  table = pd.read_html(response.text)


In [13]:
tickers

['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AMD',
 'AES',
 'AFL',
 'A',
 'APD',
 'ABNB',
 'AKAM',
 'ALB',
 'ARE',
 'ALGN',
 'ALLE',
 'LNT',
 'ALL',
 'GOOGL',
 'GOOG',
 'MO',
 'AMZN',
 'AMCR',
 'AEE',
 'AEP',
 'AXP',
 'AIG',
 'AMT',
 'AWK',
 'AMP',
 'AME',
 'AMGN',
 'APH',
 'ADI',
 'AON',
 'APA',
 'APO',
 'AAPL',
 'AMAT',
 'APP',
 'APTV',
 'ACGL',
 'ADM',
 'ARES',
 'ANET',
 'AJG',
 'AIZ',
 'T',
 'ATO',
 'ADSK',
 'ADP',
 'AZO',
 'AVB',
 'AVY',
 'AXON',
 'BKR',
 'BALL',
 'BAC',
 'BAX',
 'BDX',
 'BBY',
 'TECH',
 'BIIB',
 'BLK',
 'BX',
 'XYZ',
 'BK',
 'BA',
 'BKNG',
 'BSX',
 'BMY',
 'AVGO',
 'BR',
 'BRO',
 'BLDR',
 'BG',
 'BXP',
 'CHRW',
 'CDNS',
 'CPT',
 'CPB',
 'COF',
 'CAH',
 'CCL',
 'CARR',
 'CVNA',
 'CAT',
 'CBOE',
 'CBRE',
 'CDW',
 'COR',
 'CNC',
 'CNP',
 'CF',
 'CRL',
 'SCHW',
 'CHTR',
 'CVX',
 'CMG',
 'CB',
 'CHD',
 'CIEN',
 'CI',
 'CINF',
 'CTAS',
 'CSCO',
 'C',
 'CFG',
 'CLX',
 'CME',
 'CMS',
 'KO',
 'CTSH',
 'COIN',
 'CL',
 'CMCSA',
 'FIX',
 'CAG',
 'COP',
 'ED',
 'STZ',


In [14]:
tickers = get_sp500_tickers()

for ticker in tickers:
    try:
        df = yf.download(
            tickers=ticker,
            period="max",
            interval="1d",
            auto_adjust=True,
            progress=False
        )

        if df.empty:
            continue

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.droplevel('Ticker')

        df = df.sort_index()
        df = df[~df.index.duplicated(keep="first")]

        # --- keep only last 40 years if more than that ---
        # Require at least 40 years of history (based on full data)
        last_date = df.index[-1]
        cutoff = last_date - pd.DateOffset(years=40)

        # If the series starts after the cutoff, it has < 40y -> discard
        if df.index[0] > cutoff:
            span_years = (df.index[-1] - df.index[0]).days / 365.25
            print(f"Skipping {ticker}: only {span_years:.2f} years, {len(df)} rows")
            continue

        # Otherwise keep only the last 40 years
        df = df.loc[df.index >= cutoff].copy()

        # Optional: enforce a minimum number of trading days in the kept 40y window
        # print("Minimum Trading days:", MIN_TRADING_DAYS)
        if len(df) < MIN_TRADING_DAYS-10:
            print(f"Skipping {ticker}: insufficient rows in last 40y ({len(df)} rows)")
            continue

        data_standardized, dates = transformation(df)

        columns = ['Open', 'High', 'Low', 'Close', 'Volume']
        df_processed = pd.DataFrame(data_standardized, columns=columns)
        df_processed.insert(0, "Date", dates)

        # Use dates (post-transform) for filename bounds
        first_day = dates[0].strftime("%Y-%m-%d")
        last_day  = dates[-1].strftime("%Y-%m-%d")

        file_path = os.path.join(output_dir, f"{ticker}_{first_day}_{last_day}_processed.csv")
        df_processed.to_csv(file_path, index=False)

    except Exception as e:
        print(f"Failed to process {ticker}: {e}")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4936\631958006.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  table = pd.read_html(response.text)


10080
10080
10080
Skipping ABBV: only 13.19 years, 3318 rows
Skipping ACN: only 24.65 years, 6198 rows
Skipping ADBE: only 39.58 years, 9971 rows
10080
Skipping AES: only 34.71 years, 8740 rows
10080
Skipping A: only 26.31 years, 6617 rows
10080
Skipping ABNB: only 5.25 years, 1318 rows
Skipping AKAM: only 26.37 years, 6631 rows
Skipping ALB: only 32.05 years, 8067 rows
Skipping ARE: only 28.79 years, 7243 rows
Skipping ALGN: only 25.11 years, 6316 rows
Skipping ALLE: only 12.31 years, 3096 rows
10080
Skipping ALL: only 32.77 years, 8250 rows
Skipping GOOGL: only 21.56 years, 5425 rows
Skipping GOOG: only 21.56 years, 5425 rows
10080
Skipping AMZN: only 28.82 years, 7251 rows
Skipping AMCR: only 13.82 years, 3476 rows
Skipping AEE: only 28.19 years, 7091 rows
10080
10080
10080
Skipping AMT: only 28.04 years, 7053 rows
Skipping AWK: only 17.88 years, 4500 rows
Skipping AMP: only 20.49 years, 5154 rows
10080
10080
Skipping APH: only 34.34 years, 8645 rows
10080
10080
10080
Skipping APO: 


1 Failed download:
['JBL']: Timeout('Failed to perform, curl: (28) Operation timed out after 10000 milliseconds with 0 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')


10080
10080
10080
Skipping JCI: only 38.45 years, 9687 rows
10080
Skipping KVUE: only 2.86 years, 716 rows
Skipping KDP: only 17.85 years, 4490 rows
Skipping KEY: only 38.35 years, 9659 rows
Skipping KEYS: only 11.39 years, 2865 rows
10080
Skipping KIM: only 34.30 years, 8635 rows
Skipping KMI: only 15.08 years, 3792 rows
Skipping KKR: only 15.66 years, 3939 rows
10080
Skipping KHC: only 10.68 years, 2688 rows
10080
10080
Skipping LH: only 35.95 years, 9054 rows
10080
Skipping LW: only 9.33 years, 2345 rows
Skipping LVS: only 21.24 years, 5343 rows
Skipping LDOS: only 19.40 years, 4880 rows
10080
Skipping LII: only 26.62 years, 6696 rows
10080
Skipping LIN: only 33.73 years, 8493 rows
Skipping LYV: only 20.22 years, 5086 rows
10080
10080
10080
Skipping LULU: only 18.63 years, 4686 rows
Skipping LYB: only 15.87 years, 3993 rows
10080
Skipping MPC: only 14.72 years, 3700 rows
Skipping MAR: only 27.97 years, 7037 rows
10080
Skipping MLM: only 32.06 years, 8069 rows
10080
Skipping MA: only